In [34]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score

import evaluate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Setup and Dataset

In [2]:
# IMDb dataset

dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [4]:
# For faster implementation, I use a subset

small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

In [5]:
# Re-map labels: 1 for positive, 0 for negative
# I'll use it later

def label_to_text(label):
    return "positive" if label==1 else "negative"

In [6]:
df = pd.DataFrame(small_train)
df

,text,label
0,There is no relation at all between Fortier an...,1
1,This movie is a great. The plot is very true t...,1
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0
3,In the process of trying to establish the audi...,1
4,"Yeh, I know -- you're quivering with excitemen...",0
...,...,...
1995,24 has got to be the best spy/adventure series...,1
1996,The third collaboration for Karloff and Lugosi...,1
1997,I bought Dark Angel seasons 1 & 2 two weeks ag...,1
1998,I shudder to think what people must have thoug...,0


In [7]:
df["label"].value_counts()

label
1    1000
0    1000
Name: count, dtype: int64

I use a reduced IMDb dataset (2000 for training, 500 for testing) for faster implementation. The training dataset is èerfectly balanced

## 2. Tokenization and pre-processing

In [8]:
# Choose tokenizer -> that of distilGPT2
# And fix padding token

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

In [9]:
# Function for pre-processing

def preprocess_function(example):
    return tokenizer(example["text"], truncation=True, pading="max_length", max_length=128)

In [10]:
# Application to the datasets

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

In [11]:
# Remove useless columns

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

In [12]:
# Rename column "label" according to what the model expects to find

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

In [13]:
# Pythorch format

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [14]:
# Check

tokenized_train[0]

{'labels': tensor(1),
 'input_ids': tensor([ 1858,   318,   645,  8695,   379,   477,  1022,  6401,   959,   290,
          4415,  5329,   475,   262,  1109,   326,  1111,   389,  1644,  2168,
           546,  6590,  6741,    13,  4415,  5329,  3073, 42807,    11,  6401,
           959,  3073,  6833,    13,  4415,  5329, 21528,   389,  2407,  2829,
            13,  6401,   959,   338,  7110,   389,  1290,   517,  8253,   986,
          6401,   959,  3073,   517,   588,  5537,  8932,   806,    11,   611,
           356,   423,   284,  4136, 20594,   986,   383,  1388,  2095,   318,
          4939,   290,  7650,    78,    11,   475,   423,   366, 27659, 40024,
           590,  1911,  4380,   588,   284,  8996,    11,   284,  5052,    11,
           284, 13446,    13,  1374,   546,   655, 13226,    30, 40473,  1517,
          1165,    11,   661,  3597,  6401,   959,  3073,  1605,   475,    11,
           319,   262,   584,  1021,    11, 11810,   484,  4702,  1605,  2168,
           357, 1

The raw text is converted into token IDs and attention masks.

## 3. Load the model and test inference

In [15]:
model_name = "distilgpt2"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|███████████████████████| 76/76 [00:00<00:00, 12408.70it/s]
GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# Fix padding

model.config.pad_token_id = tokenizer.pad_token_id

In [17]:
print(model)

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=2, bias=False)
)


In [18]:
# Test the inference

text = "This movie was absolutely fantastic!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(logits)

tensor([[ 1.6634, -2.1437]])


In [19]:
# Get probabilities

probs = torch.nn.functional.softmax(logits, dim=-1)
print(probs)

tensor([[0.9783, 0.0217]])


In [20]:
# Prediction

pred = torch.argmax(probs, dim=-1).item()
print(pred)

0


In [21]:
# Function to make predictions

def predict(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()

    return pred, probs

In [22]:
predict("This was so bad")

(0, tensor([[0.9842, 0.0158]]))

In [23]:
predict("This was incredible")

(0, tensor([[0.9860, 0.0140]]))

I initialized a pre-trained model (GPT-2) with a classification head. The model has not been fine tuned on the sentiment classification task.

## 4. LoRA (PEFT)

In [24]:
# Configure LoRA

lora_config = LoraConfig(
    r=8, # rank
    lora_alpha=16, # 2*r usually
    target_modules=["c_attn"], # apply LoRA to the attention layer
    lora_dropout=0.1, #regularization
    bias="none",
    task_type="SEQ_CLS"
)

In [25]:
# Apply to the model

model = get_peft_model(model, lora_config)

/home/asus/miniconda3/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [26]:
# Check parameters

model.print_trainable_parameters()

trainable params: 148,992 || all params: 82,063,104 || trainable%: 0.1816


Only a small fraction of the model parameters (around 18%) will be updated during training, thanks to LoRA.

## 5. Training with LoRA

In [29]:
# Configure training

training_args = TrainingArguments(
    
    output_dir="./results_lora",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",

    learning_rate=2e-4,

    report_to="none"
)

In [30]:
# Accuracy

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy. compute(predictions=preds, references=labels)

In [31]:
# Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [32]:
# Train

trainer.train()

/home/asus/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.971707,0.482072,0.834000
2,0.365750,0.467474,0.836000
3,0.294320,0.438289,0.834000


/home/asus/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/asus/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1500, training_loss=0.5398979125420252, metrics={'train_runtime': 1279.6915, 'train_samples_per_second': 4.689, 'train_steps_per_second': 1.172, 'total_flos': 196666195968000.0, 'train_loss': 0.5398979125420252, 'epoch': 3.0})

## 6. Real predictions

In [33]:
# Predictions on test set

predictions = trainer.predict(tokenized_test)
logits = predictions.predictions
preds = np.argmax(logits, axis=1)

labels = predictions.label_ids

/home/asus/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [35]:
# Accuracy and F1 score

accuracy = accuracy_score(labels, preds)
print("Accuracy: ", accuracy)

f1 = f1_score(labels, preds)
print("F1: ", f1)

Accuracy:  0.834
F1:  0.839458413926499


In [36]:
# Function to make predictions

def predict_text(text):
    model.eval()
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)    
    pred = torch.argmax(probs, dim=-1).item()
    
    label = "positive" if pred == 1 else "negative"
    confidence = probs.max().item()
    
    return label, confidence

In [37]:
sentences = [
    "This movie was absolutely amazing!",
    "I hated every second of this film.",
    "It was okay, nothing special.",
    "One of the worst movies ever made.",
    "Incredible acting and beautiful story!"
]

for s in sentences:
    label, conf = predict_text(s)
    print(f"\n\nText: {s}")
    print(f"Prediction: {label} ({conf:.2f})")



Text: This movie was absolutely amazing!
Prediction: positive (0.97)


Text: I hated every second of this film.
Prediction: negative (0.91)


Text: It was okay, nothing special.
Prediction: negative (0.89)


Text: One of the worst movies ever made.
Prediction: negative (0.92)


Text: Incredible acting and beautiful story!
Prediction: positive (0.98)


In [39]:
# Some particular sentences

sentences = [
    "It was not bad, but not great either.", # ambiguous
    "Yeah, best movie ever... I almost fell asleep.", # sarcasm
    "Great visuals but terrible story." # mixed sentiment
]

for s in sentences:
    label, conf = predict_text(s)
    print(f"\n\nText: {s}")
    print(f"Prediction: {label} ({conf:.2f})")



Text: It was not bad, but not great either.
Prediction: negative (0.72)


Text: Yeah, best movie ever... I almost fell asleep.
Prediction: negative (0.90)


Text: Great visuals but terrible story.
Prediction: negative (0.92)


Accuracy and F1 score reach good values (both around 83%). Moreover, the model converges quickly as accuracy is stable since the beginning. This is expected since few parameters are updated using LoRA.

Simple examples are well understood and predictions are correct. For the borderline cases:
* Ambiguous one is classified as negative (maybe the model is looking at the end of the sentence?)
* The sarcasm is captured!
* Also in the mixed sentiment case, the model seems to give more importance to the negative part (in the end of the sentence too).